# 04-attribution-estimation
Phase 4 for the generalized project (Places365 + BDD100K, DINOv2 deployed + ConvNeXt canary).
Attach only the **03a-embeddings** notebook output. Outputs: `detector_objects.joblib`, `phase4_models.joblib`.

**1. Setup** - load embeddings and metadata

In [1]:
import glob, time, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor as RFR, RandomForestClassifier as RFC
from sklearn.metrics import r2_score
NB = glob.glob('/kaggle/input/notebooks/aadiparkhi/03a*')[0]
BK, K = ['dino', 'convnext'], 32
rng = np.random.default_rng(0); torch.manual_seed(0)
D = {}
for d in ['places', 'bdd']:
    z = np.load(f'{NB}/feats_{d}.npz'); m = pd.read_csv(f'{NB}/meta_{d}.csv')
    D[d] = {'meta': m, **{k: {'emb': z[f'{k}_emb'].astype(np.float32), 'log': z[f'{k}_log']} for k in BK}}
print({d: len(D[d]['meta']) for d in D})

{'places': 36984, 'bdd': 36046}


**2. Helpers** - correctness per head, K-S vector, Phase 4 feature builder (batch mean/std/K-S/AE error per sensor)

In [2]:
def correct(d, k):
    m, L = D[d]['meta'], D[d][k]['log']
    return (L.argmax(1) == m.label.values) if d == 'places' else (L[:, :11].argmax(1) == m.cond_id.values)

def ks_vec(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0)

def build(d, B=100, nb=30, seed=0):
    r = np.random.default_rng(seed); m = D[d]['meta']; ok = correct(d, 'dino')
    cp = np.where((m.set == 'corrupt').values)[0]; bid, sev, cor = m.bid.values, m.severity.values, m.corruption.values
    i0 = cp[sev[cp] == 0]; ck = np.zeros(bid[cp].max() + 1); ck[bid[i0]] = ok[i0]
    half = r.permutation(bid[cp].max() + 1) % 2                      # two disjoint halves of the base images
    F = {k: [] for k in BK}; meta = []
    for c in sorted(set(cor[cp])):
        for s in sorted(set(sev[cp][cor[cp] == c])):
            pool = cp[(cor[cp] == c) & (sev[cp] == s)]
            for h in (0, 1):
                ph = pool[half[bid[pool]] == h]
                for _ in range(nb):
                    b = r.choice(ph, B)
                    for k in BK:
                        P, S, err = PP[(d, k)]; X = P[b]; kv = ks_vec(X, S)
                        F[k].append(np.concatenate([X.mean(0), X.std(0), kv, [kv.mean(), err[b].mean()]]))
                    meta.append((c, s, h, ck[bid[b]].mean() - ok[b].mean()))
    return {k: np.array(v) for k, v in F.items()}, pd.DataFrame(meta, columns=['cause', 'sev', 'split', 'dloss'])

**3. Final detectors** - seeded PCA + autoencoder per domain and sensor, keeping the fitted objects (needed by the dashboard)

In [3]:
def train_ae2(z, fit):                      # same as train_ae, but also returns the model
    torch.manual_seed(0)
    x = torch.tensor(z, dtype=torch.float32); tr = x[torch.tensor(fit)]
    ae = nn.Sequential(nn.Linear(x.shape[1], 256), nn.ReLU(), nn.Linear(256, 32), nn.ReLU(),
                       nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, x.shape[1]))
    opt = torch.optim.Adam(ae.parameters(), 1e-3)
    for _ in range(60):
        for b in torch.randperm(len(tr)).split(256):
            opt.zero_grad(); ((ae(tr[b]) - tr[b]) ** 2).mean().backward(); opt.step()
    with torch.no_grad(): return ae, ((ae(x) - x) ** 2).mean(1).numpy()

PP, OBJ = {}, {}
for dd, k in [(a, b) for a in D for b in BK]:
    mm, e = D[dd]['meta'], D[dd][k]['emb']; vv = (mm.set == 'val').values
    mu, sd = e[vv].mean(0), e[vv].std(0) + 1e-6
    z = (e - mu) / (sd)
    pca = PCA(K, random_state=0).fit(z[vv]); P = pca.transform(z)
    ae, err = train_ae2(z, mm.set.isin(['val', 'train']).values)
    OBJ[(dd, k)] = dict(mu=mu, sd=sd, pca=pca, ae=ae.state_dict(), dim=z.shape[1])
    PP[(dd, k)] = (P, np.sort(P[vv], 0), err)
    OBJ[(dd, k)]['S'] = PP[(dd, k)][1]          # sorted val projections (reference for K-S)
    print(dd, k, "val AE error:", round(err[vv].mean(), 4))

joblib.dump(OBJ, '/kaggle/working/detector_objects.joblib')

places dino val AE error: 0.4048
places convnext val AE error: 0.4516
bdd dino val AE error: 0.3838
bdd convnext val AE error: 0.4494


['/kaggle/working/detector_objects.joblib']

**4. BDD condition-mix setup** - reference mix, group shifts, batch sampler

In [4]:
NAMES = ['clear_day', 'clear_night', 'clear_twilight', 'overcast_day', 'overcast_twilight', 'rainy_day', 'rainy_night',
         'rainy_twilight', 'snow_fog_day', 'snow_fog_night', 'snow_fog_twilight']
GROUPS = {'night': lambda n: n.endswith('night'), 'day': lambda n: n.endswith('_day'), 'twilight': lambda n: n.endswith('twilight'),
          'rain': lambda n: n.startswith('rainy'), 'snow_fog': lambda n: n.startswith('snow_fog'), 'overcast': lambda n: n.startswith('overcast')}
d = 'bdd'; m = D[d]['meta']; cond = m.cond_id.values; ok = correct(d, 'dino')
pref = np.bincount(cond[(m.set == 'val').values], minlength=11) / (m.set == 'val').sum()   # reference mix = clean val
tst = np.where((m.set == 'test').values)[0]; rng.shuffle(tst); h = len(tst) // 2
PA, PB = tst[:h], tst[h:]

def draw(pool, p, B=100):
    cl = rng.choice(11, B, p=p); idx = np.empty(B, int)
    for c in range(11):
        w = np.where(cl == c)[0]
        if len(w): idx[w] = rng.choice(pool[cond[pool] == c], len(w))
    return idx

acc0 = np.array([ok[draw(PB, pref)].mean() for _ in range(400)])      # reference-mix batch accuracy
print('reference batch accuracy:', round(acc0.mean(), 3))

reference batch accuracy: 0.729


**5. Label-shift (BBSE-style) impact estimate** - per-class accuracy and confusion matrix from clean val

In [5]:
from scipy.optimize import nnls

lg = D[d]['dino']['log'][:, :11]            # 11 condition logits (first 11 of 13)
pred = lg.argmax(1)
v = (m.set == 'val').values

# C[i, j] = P(predicted class i | true class j), from clean val
C = np.zeros((11, 11))
for j in range(11):
    r = v & (cond == j)
    if r.sum(): C[:, j] = np.bincount(pred[r], minlength=11) / r.sum()
acc_c = np.diag(C)                           # per-class accuracy
print("per-class val acc:", dict(zip(NAMES, acc_c.round(2))))

def est_change(b):
    q = np.bincount(pred[b], minlength=11) / len(b)     # predicted-class histogram
    w, _ = nnls(C, q); w /= w.sum()                     # estimated true mix
    return w @ acc_c - pref @ acc_c

res = []
for g, f in GROUPS.items():
    u = np.array([float(f(n)) for n in NAMES]); u /= u.sum()
    for lam in (0.25, 0.5, 0.75):
        bs = [draw(PB, (1 - lam) * pref + lam * u) for _ in range(30)]
        res.append(dict(group=g, lam=lam,
                        acc_change=np.mean([ok[b].mean() for b in bs]) - acc0.mean(),
                        pred_bbse=np.mean([est_change(b) for b in bs])))
print(pd.DataFrame(res).round(3).to_string(index=False))

per-class val acc: {'clear_day': np.float64(0.78), 'clear_night': np.float64(0.85), 'clear_twilight': np.float64(0.66), 'overcast_day': np.float64(0.74), 'overcast_twilight': np.float64(0.46), 'rainy_day': np.float64(0.74), 'rainy_night': np.float64(0.74), 'rainy_twilight': np.float64(0.3), 'snow_fog_day': np.float64(0.85), 'snow_fog_night': np.float64(0.84), 'snow_fog_twilight': np.float64(0.56)}
   group  lam  acc_change  pred_bbse
   night 0.25       0.024      0.022
   night 0.50       0.045      0.038
   night 0.75       0.063      0.062
     day 0.25      -0.005      0.008
     day 0.50       0.015      0.014
     day 0.75       0.003      0.030
twilight 0.25      -0.027     -0.064
twilight 0.50      -0.078     -0.111
twilight 0.75      -0.141     -0.158
    rain 0.25      -0.032     -0.007
    rain 0.50      -0.073     -0.037
    rain 0.75      -0.098     -0.070
snow_fog 0.25      -0.002     -0.001
snow_fog 0.50       0.025      0.000
snow_fog 0.75       0.049     -0.007
overcas

**6. Loss regressor + attribution on corruptions**, then the failure case on composition shifts

In [6]:
F, mt = build(d)
XX = {'dino': F['dino'], 'both': np.hstack([F[k] for k in F])}
s0, s1, nc = (mt.split == 0).values, (mt.split == 1).values, (mt.cause != 'clean').values
reg = {k: RandomForestRegressor(300, n_jobs=-1, random_state=0).fit(XX[k][s0], mt.dloss[s0]) for k in XX}
att = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(XX['both'][s0 & nc], mt.cause[s0 & nc])
print("held-out loss R2:", {k: round(r2_score(mt.dloss[s1], reg[k].predict(XX[k][s1])), 3) for k in XX})

def feat(b):
    out = []
    for k in F:
        P, S, err = PP[(d, k)]; x = P[b]; kv = ks_vec(x, S)
        out.append(np.concatenate([x.mean(0), x.std(0), kv, [kv.mean(), err[b].mean()]]))
    return out

res = []
for g, f in GROUPS.items():
    u = np.array([float(f(n)) for n in NAMES]); u /= u.sum()
    for lam in (0.25, 0.5, 0.75):
        bs = [draw(PB, (1 - lam) * pref + lam * u) for _ in range(30)]
        fs = [feat(b) for b in bs]
        Xd, Xb = np.array([z[0] for z in fs]), np.array([np.concatenate(z) for z in fs])
        res.append(dict(group=g, lam=lam, acc_change=np.mean([ok[b].mean() for b in bs]) - acc0.mean(),
                        pred_dino=-reg['dino'].predict(Xd).mean(), pred_both=-reg['both'].predict(Xb).mean(),
                        attributed=pd.Series(att.predict(Xb)).mode()[0]))
print(pd.DataFrame(res).round(3).to_string(index=False))

held-out loss R2: {'dino': 0.834, 'both': 0.837}
   group  lam  acc_change  pred_dino  pred_both attributed
   night 0.25       0.015     -0.002      0.000 brightness
   night 0.50       0.030     -0.099     -0.095 brightness
   night 0.75       0.054     -0.125     -0.114 brightness
     day 0.25      -0.001     -0.008     -0.002 brightness
     day 0.50       0.023     -0.016     -0.007 brightness
     day 0.75       0.008     -0.041     -0.038 brightness
twilight 0.25      -0.034     -0.005     -0.001 brightness
twilight 0.50      -0.080     -0.016     -0.010 brightness
twilight 0.75      -0.134     -0.093     -0.058 brightness
    rain 0.25      -0.038     -0.003      0.001 brightness
    rain 0.50      -0.070     -0.004     -0.001 brightness
    rain 0.75      -0.102     -0.011     -0.006 brightness
snow_fog 0.25       0.010     -0.003      0.001 brightness
snow_fog 0.50       0.036     -0.007     -0.004 brightness
snow_fog 0.75       0.043     -0.138     -0.109 brightness
overcas

**7. Router** - `mix_shift` attribution class, leave-one-group-out check

In [7]:
def mixset(pool, n, lams=(0.25, 0.5, 0.75)):
    out = []
    for g, f in GROUPS.items():
        u = np.array([float(f(nm)) for nm in NAMES]); u /= u.sum()
        for lam in lams:
            for _ in range(n):
                b = draw(pool, (1 - lam) * pref + lam * u)
                out.append((g, np.concatenate(feat(b)), ok[b].mean() - acc0.mean(), est_change(b)))
    return out

MA, MB = mixset(PA, 20), mixset(PB, 20)          # train batches from one test half, eval from the other
Xc, yc = XX['both'][s0 & nc], mt.cause[s0 & nc].values
Xa, ga = np.array([r[1] for r in MA]), np.array([r[0] for r in MA])
fit2 = lambda X, y: RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(X, y)

rows = []
for g in GROUPS:                                  # leave one group out
    k = ga != g
    m2 = fit2(np.vstack([Xc, Xa[k]]), np.concatenate([yc, np.full(k.sum(), 'mix_shift')]))
    te_ = [r for r in MB if r[0] == g]
    Xt, true, bb = np.array([r[1] for r in te_]), np.array([r[2] for r in te_]), np.array([r[3] for r in te_])
    rg = -reg['both'].predict(Xt); mix = m2.predict(Xt) == 'mix_shift'
    rows.append(dict(group=g, routed_mix=mix.mean(), mae_reg=abs(rg - true).mean(),
                     mae_bbse=abs(bb - true).mean(), mae_router=abs(np.where(mix, bb, rg) - true).mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))

att2 = fit2(np.vstack([Xc, Xa]), np.concatenate([yc, np.full(len(Xa), 'mix_shift')]))
pc = att2.predict(XX['both'][s1 & nc])
print("corruption attribution acc:", round((pc == mt.cause[s1 & nc].values).mean(), 3),
      "| misrouted to mix_shift:", round((pc == 'mix_shift').mean(), 3))

   group  routed_mix  mae_reg  mae_bbse  mae_router
   night         1.0    0.124     0.033       0.033
     day         1.0    0.051     0.039       0.039
twilight         1.0    0.064     0.052       0.052
    rain         1.0    0.079     0.056       0.056
snow_fog         1.0    0.080     0.046       0.046
overcast         1.0    0.066     0.050       0.050
corruption attribution acc: 0.987 | misrouted to mix_shift: 0.011


**8. Final fit and save** - loss regressors and attribution (BDD includes `mix_shift`) for both domains

In [8]:
ART = dict(reg={}, att={}, C=C, acc_c=acc_c, pref=pref)
for dd in ('places', 'bdd'):
    F_, mt_ = build(dd); X = {'dino': F_['dino'], 'both': np.hstack([F_[k] for k in BK])}
    a0, a1, nc_ = (mt_.split == 0).values, (mt_.split == 1).values, (mt_.cause != 'clean').values
    print(dd, "held-out loss R2:", {k: round(r2_score(mt_.dloss[a1], RFR(300, n_jobs=-1, random_state=0).fit(X[k][a0], mt_.dloss[a0]).predict(X[k][a1])), 3) for k in X})
    chk = RFC(300, n_jobs=-1, random_state=0).fit(X['both'][a0 & nc_], mt_.cause[a0 & nc_].values)
    print(dd, "held-out attribution acc:", round(chk.score(X['both'][a1 & nc_], mt_.cause[a1 & nc_].values), 3))
    ART['reg'][dd] = {k: RFR(300, n_jobs=-1, random_state=0).fit(X[k], mt_.dloss) for k in X}
    Xa, ya = X['both'][nc_], mt_.cause[nc_].values
    if dd == 'bdd':                                   # add the mix_shift class from fresh mix-shift batches
        MX = mixset(PA, 20) + mixset(PB, 20)
        Xm = np.array([r[1] for r in MX])
        Xa, ya = np.vstack([Xa, Xm]), np.concatenate([ya, np.full(len(Xm), 'mix_shift')])
    ART['att'][dd] = RFC(300, n_jobs=-1, random_state=0).fit(Xa, ya)

joblib.dump(ART, '/kaggle/working/phase4_models.joblib')
print("saved:", list(ART))

places held-out loss R2: {'dino': 0.88, 'both': 0.893}
places held-out attribution acc: 0.961
bdd held-out loss R2: {'dino': 0.834, 'both': 0.837}
bdd held-out attribution acc: 0.999
saved: ['reg', 'att', 'C', 'acc_c', 'pref']
